In [1]:
import numpy as np
from pathlib import Path
import pandas as pd
try:
    from Preprocessing.analysis_utils import get_stimulus_timestamps, get_running_timestamps, get_running_df, resample_dff_for_trial, resample_running_for_trial
except ModuleNotFoundError:
    from analysis_utils import get_stimulus_timestamps, get_running_timestamps, get_running_df, resample_dff_for_trial, resample_running_for_trial
from hdmf_zarr.nwb import NWBZarrIO
import matplotlib.pyplot as plt
import anndata as ad
import pickle
from scipy.interpolate import interp1d
from scipy.optimize import curve_fit
from pathlib import Path
import os
import sys
import time
import zarr
import numcodecs
import shutil
import re
RESAMPLE_RATE = 10    # Hz
PRE_STIM      = 1.0   # seconds before stimulus onset
POST_STIM     = 1.0   # seconds after stimulus offset

SCRATCH_DIR = Path("/root/capsule/scratch")

DATA_DIR = Path("/root/capsule/data")
STIM_ALIGNED_DIR = SCRATCH_DIR / "ophys" / "stim_aligned"
MORPH_DIR = SCRATCH_DIR / "morphology"
TRANSCRIPT_DIR = SCRATCH_DIR / 'transcriptomics'


MULTIMODAL = SCRATCH_DIR / "multimodal_data"
MULTIMODAL.mkdir(parents=True, exist_ok=True)

# Import tuning and GLM functions from code/functions/
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('.'), '..')))
from functions.tuning import compute_tuning_for_session, save_tuning_to_zarr
from functions.glm import add_glm_aggregate_columns

load data information

In [ ]:
data_info = pickle.load(open('data_info_hcr.pkl', 'rb'))
mouse_ids = list(data_info.keys())
print('mouse_ids:', mouse_ids)

mouse_ids: [755252, 767018, 767022, 782149, 788406, 790322]


In [ ]:
glm_keys = ['alpha', 'coef', 'score', 'y', 'y_hat']
STIM_ALIGNED = SCRATCH_DIR / "stim_aligned_dff"


def _safe_create_dataset(group, name, max_retries=5, backoff=0.5, **kwargs):
    """Retry group.create_dataset on transient FileNotFoundError (zarr partial-rename races).
    Also handle zarr.errors.ContainsArrayError by removing the conflicting array and retrying."""
    for attempt in range(1, max_retries + 1):
        try:
            return group.create_dataset(name, **kwargs)
        except FileNotFoundError:
            if attempt == max_retries:
                raise
            time.sleep(backoff * attempt)
        except zarr.errors.ContainsArrayError:
            # If a zarr Array exists where a Group is expected for the path component,
            # remove the conflicting array so we can recreate the intended dataset.
            try:
                del group[name]
            except Exception:
                # best-effort removal; we'll retry until max_retries
                pass
            if attempt == max_retries:
                raise
            time.sleep(backoff * attempt)

for mi, mouse_id in enumerate(mouse_ids):
    out_path = MULTIMODAL / f"{mouse_id}h_multimodal_data.zarr"
    print(f"\n{'='*60}")
    print(f"[{mi+1}/{len(mouse_ids)}] Processing mouse {mouse_id}")
    print(f"{'='*60}")
    

    cell_types_coreg_df = pd.read_csv(TRANSCRIPT_DIR / f'{mouse_id}h_cell_types_coreg.csv')
    cellxgene_coreg_df = pd.read_csv(TRANSCRIPT_DIR / f'{mouse_id}h_cellxgene_coreg.csv')
    
    unique_ids = cell_types_coreg_df['unique_id'].values


    shared_columns = [col for col in cell_types_coreg_df.columns if col in cellxgene_coreg_df.columns]
    cellxgene_columns = [col for col in cellxgene_coreg_df.columns if col not in shared_columns]
    cell_type_columns = [col for col in cell_types_coreg_df.columns if col not in shared_columns]

    cell_types_table = cell_types_coreg_df.set_index('unique_id')[cell_type_columns]
    cellxgene_table = cellxgene_coreg_df.set_index('unique_id')[cellxgene_columns]

    # ── Open or create Zarr store ───────────────────────────────────
    store = zarr.DirectoryStore(str(out_path))
    if out_path.exists():
        print("  → Existing Zarr store found – resuming...")
        root = zarr.open_group(store, mode='a')
    else:
        print("  → Creating new Zarr store...")
        root = zarr.group(store, overwrite=True)

    # ── Save unique_id ──────────────────────────────────────────────
    if 'unique_id' in root:
        print("  → unique_id already exists – skipping.")
    else:
        print("  → Saving unique_id array...")
        _ = root.array('unique_id', data=unique_ids, dtype=str,
                   object_codec=numcodecs.VLenUTF8())

    # ── Save transcriptomics tables ─────────────────────────────────
    if 'transcriptomics' not in root:
        root.create_group('transcriptomics')
    transcriptomics_group: zarr.Group = root['transcriptomics']  # type: ignore[assignment]

    # cell_type
    if 'cell_type' in transcriptomics_group:
        print("  → transcriptomics/cell_type already exists – skipping.")
    else:
        print("  → Saving transcriptomics / cell_type...")
        ct_group = transcriptomics_group.create_group('cell_type')
        ct_group.attrs['index'] = list(cell_types_table.index)
        ct_group.attrs['columns'] = list(cell_types_table.columns)
        for col in cell_types_table.columns:
            vals = cell_types_table[col].values
            if vals.dtype == object:
                _ = ct_group.array(str(col), data=vals, dtype=str,
                               object_codec=numcodecs.VLenUTF8())
            else:
                _ = ct_group.array(str(col), data=vals)

    # cellxgene
    if 'cellxgene' in transcriptomics_group:
        print("  → transcriptomics/cellxgene already exists – skipping.")
    else:
        print("  → Saving transcriptomics / cellxgene...")
        cxg_group = transcriptomics_group.create_group('cellxgene')
        cxg_group.attrs['index'] = list(cellxgene_table.index)
        cxg_group.attrs['columns'] = list(cellxgene_table.columns)
        for col in cellxgene_table.columns:
            vals = cellxgene_table[col].values
            if vals.dtype == object:
                _ = cxg_group.array(str(col), data=vals, dtype=str,
                                object_codec=numcodecs.VLenUTF8())
            else:
                _ = cxg_group.array(str(col), data=vals)

    # ── Save morphology tables ─────────────────────────────────
    if 'morphology' not in root:
        root.create_group('morphology')
    morph_group = root['morphology']  # type: ignore[assignment]

    if 'mask_properties' in morph_group:
        print("  → morphology/mask_properties already exists – skipping.")
    else:
        print("  → Adding morphology / mask_properties...")
        mask_prop_group = morph_group.create_group('mask_properties')

        morph_csv = MORPH_DIR / f"{mouse_id}h_mask_properties.csv"
        print(f"        Loading morphology file: {morph_csv.name}...")
        try:
            df_masks_props = pd.read_csv(morph_csv)
        except FileNotFoundError:
            print(f"        → File not found: {morph_csv} — skipping morphology for mouse {mouse_id}.")
        else:
            print("        Subsetting mask properties to coregistered cells...")
        coreg_mask = (cell_types_coreg_df['mouse_id'] == mouse_id)
        mask_labels = cell_types_coreg_df.loc[coreg_mask, 'cell_id - zstack'].values
        unique_id = cell_types_coreg_df.loc[coreg_mask, 'unique_id'].values
        df_masks_props = df_masks_props.set_index('label').loc[mask_labels]
        df_masks_props['unique_id'] = unique_id
        df_masks_props.set_index('unique_id', inplace=True, drop=True)

        print("        Writing mask properties to Zarr...")
        mask_prop_group.attrs['index'] = list(df_masks_props.index)
        mask_prop_group.attrs['columns'] = list(df_masks_props.columns)
        mask_prop_group.attrs['n_rows'] = len(df_masks_props)
        for col in df_masks_props.columns:
            vals = df_masks_props[col].values
            if vals.dtype == object:
                _ = mask_prop_group.array(str(col), data=vals, dtype=str,
                            object_codec=numcodecs.VLenUTF8())
            else:
                _ = mask_prop_group.array(str(col), data=vals)


    # ── Save ophys data per session × stimulus ──────────────────────
    if 'ophys' not in root:
        root.create_group('ophys')
    ophys_group: zarr.Group = root['ophys']  # type: ignore[assignment]

    if 'drifting_gratings' not in ophys_group:
        ophys_group.create_group('drifting_gratings')
    dg_group: zarr.Group = ophys_group['drifting_gratings']  # type: ignore[assignment]

    for si, session_name in enumerate(['session_1', 'session_2', 'session_3']):
        print(f"  → [{si+1}/3] Processing {session_name}...")

        if session_name not in dg_group:
            dg_group.create_group(session_name)
        sess_group: zarr.Group = dg_group[session_name]  # type: ignore[assignment]

        # ── stim_aligned_dff ────────────────────────────────────────
        if 'stim_aligned_dff' not in sess_group:
            sess_group.create_group('stim_aligned_dff')
        sadff_group: zarr.Group = sess_group['stim_aligned_dff']  # type: ignore[assignment]

        in_path = STIM_ALIGNED_DIR / f"{mouse_id}_{session_name}.pkl"

        # Load session data
        print(f"      → Loading {in_path.name}...")
        with open(in_path, 'rb') as f:
            session_data = pickle.load(f)

        stim_names: list[str] = list(session_data.keys())

        for sti, stim_name in enumerate(stim_names):
            if stim_name in sadff_group:
                # Quick check: does the group already have 'dff'?
                stim_grp_existing: zarr.Group = sadff_group[stim_name]  # type: ignore[assignment]
                if 'dff' in stim_grp_existing and 'trial_info' in stim_grp_existing:
                    print(f"      → [{sti+1}/{len(stim_names)}] Stimulus {stim_name} already exists – skipping.")
                    continue

            print(f"      → [{sti+1}/{len(stim_names)}] Stimulus: {stim_name}")
            sd: dict = session_data
            planes: list[str] = list(sd[stim_name]['dff'].keys())
            all_dff: list[np.ndarray] = []
            all_unique_ids_list: list[np.ndarray] = []

            for plane in planes:
                cell_ids_all = sd[stim_name]['dff'][plane]['cell_ids']
                coreg_mask = (
                    (cell_types_coreg_df['mouse_id'] == mouse_id) &
                    (cell_types_coreg_df['plane'] == plane)
                )
                cell_ids_coreg: np.ndarray = cell_types_coreg_df.loc[
                    coreg_mask, f'cell_id - {session_name}'
                ].values
                unique_ids_plane: np.ndarray = cell_types_coreg_df.loc[
                    coreg_mask, 'unique_id'
                ].values

                cell_id_locs = np.array([
                    np.argwhere(cell_ids_all == cid)[0, 0]
                    for cid in cell_ids_coreg
                ]).astype(int)

                all_dff.append(
                    sd[stim_name]['dff'][plane]['data'][..., cell_id_locs]
                )
                all_unique_ids_list.append(unique_ids_plane)

            all_unique_ids: np.ndarray = np.concatenate(all_unique_ids_list, axis=-1)
            dff_coreg: np.ndarray = np.concatenate(all_dff, axis=-1)[
                ..., np.argwhere(np.isin(unique_ids, all_unique_ids))[:, 0]
            ]

            # Write stimulus group (remove partial if exists)
            if stim_name in sadff_group:
                del sadff_group[stim_name]
            stim_group = sadff_group.create_group(stim_name)

            # print(f"        Writing dff {dff_coreg.shape}...")
            # _ = stim_group.create_dataset(
            #     'dff', data=dff_coreg, dtype='float32',
            #     chunks=(1, dff_coreg.shape[1], min(int(dff_coreg.shape[2]), 64)),
            #     compressor=numcodecs.Blosc(cname='zstd', clevel=3),
            # )

            print(f"        Writing dff {dff_coreg.shape}...")
            _ = _safe_create_dataset(
                stim_group, 'dff', data=dff_coreg, dtype='float32',
                chunks=(1, dff_coreg.shape[1], min(int(dff_coreg.shape[2]), 64)),
                compressor=numcodecs.Blosc(cname='zstd', clevel=3),
            )

            print(f"        Writing running...")
            running_data = sd[stim_name]['running']
            _ = stim_group.array(
                'running',
                data=running_data,
                dtype='float32',
                chunks=(1, running_data.shape[1], 2),
                compressor=numcodecs.Blosc(cname='zstd', clevel=3),
            )

            _ = stim_group.array(
                'time_relative',
                data=sd[stim_name]['time_relative'],
            )

            print(f"        Writing trial_info...")
            trial_info: pd.DataFrame = sd[stim_name]['trial_info']
            ti_group = stim_group.create_group('trial_info')
            ti_group.attrs['columns'] = list(trial_info.columns)
            ti_group.attrs['n_rows'] = len(trial_info)
            for col in trial_info.columns:
                vals = trial_info[col].values
                if vals.dtype == object:
                    str_vals = np.array(['' if pd.isna(v) else str(v) for v in vals], dtype=object)
                    _ = ti_group.array(str(col), data=str_vals, dtype=str,
                        object_codec=numcodecs.VLenUTF8())
                else:
                    _ = ti_group.array(str(col), data=vals)
        # ── GLM data ───────────────────────────────────────────────
        print(f"  → adding glm results from {session_name}...")
        if 'glm' not in sess_group:
            sess_group.create_group('glm')
        glm_group: zarr.Group = sess_group['glm']  # type: ignore[assignment]

        glm_path = DATA_DIR / data_info[mouse_id]['ophys'][session_name]['glm']
        glm_files: list[Path] = list(glm_path.glob('*.hdf'))

        # Build df for cell_id mapping (needed for GLM)
        df: pd.DataFrame = cell_types_coreg_df.set_index('unique_id')
        # Compute unique_ids for this mouse
        mouse_unique_ids: np.ndarray = cell_types_coreg_df.loc[
            cell_types_coreg_df['mouse_id'] == mouse_id, 'unique_id'
        ].values
        
        for ki, key in enumerate(glm_keys):
            if key in glm_group:
                # Check it has at least one array inside
                key_grp_existing: zarr.Group = glm_group[key]  # type: ignore[assignment]
                if len(list(key_grp_existing.arrays())) > 0:
                    print(f"        GLM key '{key}' already exists – skipping.")
                    continue

            print(f"        [{ki+1}/{len(glm_keys)}] GLM key: {key}")
            glm_output_parts: list[pd.DataFrame] = []
            for glm_file in glm_files:
                glm_output_parts.append(pd.read_hdf(str(glm_file), key=key))
            glm_output_total: pd.DataFrame = pd.concat(glm_output_parts, axis=0)
            glm_output_total = glm_output_total.set_index(glm_output_total.index)

            cell_ids: list[str] = [
                f"{row['plane']}_{int(row[f'cell_id - {session_name}'])-1}"
                for _, row in df.loc[mouse_unique_ids].iterrows()
            ]
            glm_output_total = glm_output_total.loc[cell_ids].reset_index(drop=True)
            # Remove partial if exists
            if key in glm_group:
                del glm_group[key]
            key_group = glm_group.create_group(key)

            key_group.attrs['columns'] = list(glm_output_total.columns)
            key_group.attrs['n_rows'] = len(glm_output_total)
            for col in glm_output_total.columns:
                vals = glm_output_total[col].values
                if vals.dtype == object:
                    first_valid = next((v for v in vals if v is not None), None)
                    if first_valid is not None and isinstance(first_valid, (list, np.ndarray)):
                        stacked: np.ndarray = np.array(
                            [np.asarray(v, dtype=np.float32) for v in vals]
                        )
                        _ = key_group.array(str(col), data=stacked, dtype='float32')
                    else:
                        str_vals: np.ndarray = np.array(
                            [str(v) for v in vals], dtype=object
                        )
                        _ = key_group.array(str(col), data=str_vals, dtype=str,
                                      object_codec=numcodecs.VLenUTF8())
                else:
                    _ = key_group.array(str(col), data=vals)
        


    # ── Compute tuning properties per session ───────────────────────
    n_cells = len(unique_ids)
    for session_name in ['session_1', 'session_2', 'session_3']:
        sess_group = root['ophys/drifting_gratings'][session_name]

        if 'tuning_properties' in sess_group:
            print(f"  → tuning_properties/{session_name} already exists – skipping.")
            continue

        print(f"  → Computing tuning_properties for {session_name}...")
        gs = sess_group['stim_aligned_dff/GratingStim']
        dff = gs['dff'][:]
        time_rel = gs['time_relative'][:]
        ti_dict = {k: gs[f'trial_info/{k}'][:] for k in gs['trial_info'].keys()}
        trial_info = pd.DataFrame(ti_dict)

        tuning = compute_tuning_for_session(dff, trial_info, time_rel, n_cells)
        save_tuning_to_zarr(root, session_name, tuning)
        print(f"        {n_cells} cells, OSI median={tuning['OSI'].median():.3f}, "
              f"C50 non-NaN={tuning['C50'].notna().sum()}")

    # ── Compute GLM aggregate columns per session ───────────────────
    for session_name in ['session_1', 'session_2', 'session_3']:
        glm_path = f'ophys/drifting_gratings/{session_name}/glm'
        if glm_path not in root:
            print(f"  → GLM aggregates/{session_name}: no GLM group – skipping.")
            continue

        glm_grp = root[glm_path]
        coef_grp = glm_grp.get('coef', None)
        if coef_grp is None:
            continue

        # Check if aggregates already exist
        existing_keys = list(coef_grp.keys())
        if any(k.startswith('glm-coef_') for k in existing_keys):
            print(f"  → GLM aggregates/{session_name} already exist – skipping.")
            continue

        print(f"  → Computing GLM aggregate columns for {session_name}...")
        n_written = add_glm_aggregate_columns(glm_grp)
        print(f"        Wrote {n_written} aggregate arrays.")

    zarr.consolidate_metadata(store)
    print(f"  ✓ {out_path.name} saved.")

print("\n✅ All coregistered files saved as Zarr.")


[1/6] Processing mouse 755252
  → Existing Zarr store found – resuming...
  → unique_id already exists – skipping.
  → transcriptomics/cell_type already exists – skipping.
  → transcriptomics/cellxgene already exists – skipping.
  → morphology/mask_properties already exists – skipping.
  → [1/3] Processing session_1...
      → Loading 755252_session_1.pkl...
      → [1/3] Stimulus: GrayScreen
        Writing dff (2, 3624, 190)...
        Writing running...
        Writing trial_info...
      → [2/3] Stimulus: GratingStim
        Writing dff (2186, 41, 190)...
        Writing running...
        Writing trial_info...
      → [3/3] Stimulus: Catch
        Writing dff (54, 41, 190)...
        Writing running...
        Writing trial_info...
  → adding glm results from session_1...
        [1/5] GLM key: alpha
        [2/5] GLM key: coef
        [3/5] GLM key: score
        [4/5] GLM key: y
        [5/5] GLM key: y_hat
  → [2/3] Processing session_2...
      → Loading 755252_session_2.pkl.

/code/functions/tuning.py:113: OptimizeWarning: Covariance of the parameters could not be estimated
  popt, _ = curve_fit(


        190 cells, OSI median=0.094, C50 non-NaN=182
  → Computing tuning_properties for session_2...


/code/functions/tuning.py:113: OptimizeWarning: Covariance of the parameters could not be estimated
  popt, _ = curve_fit(


        190 cells, OSI median=0.087, C50 non-NaN=189
  → Computing tuning_properties for session_3...
